<a href="https://colab.research.google.com/github/yilmajung/LLM_POC_Study_2025_v2/blob/main/r2_finetune_llm_homosex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers accelerate peft sentencepiece pandas pyarrow

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, json, math, random
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

In [22]:
# Set up paths
CS_CSV    = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/gss_abt_cs_homosex.csv"       # cross-sectional long
PANEL_CSV = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/gss_abt_panel_homosex.csv"    # panel long
OUT_DIR   = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_r_trial_homosex_g4"          # where I write JSONLs & checkpoints
os.makedirs(OUT_DIR, exist_ok=True)

# LLM choice
BASE_MODEL_NAME = "meta-llama/llama-3.1-8b"  # or "mistralai/Mistral-7B-v0.3"

# Canonical bins (K=4)
ABORT4 = ["strong_anti", "anti", "pro", "strong_pro"]
TRUST3 = ["distrust", "depends", "trust"]
ENVIR3 = ["too_little", "about_right", "too_much"]
HOMOSEX4 = ["always_wrong", "almost_always_wrong", "sometimes_wrong", "not_wrong_at_all"]
ABORT2ID = {c:i for i,c in enumerate(ABORT4)}
TRUST2ID = {c:i for i,c in enumerate(TRUST3)}
ENVIR2ID = {c:i for i,c in enumerate(ENVIR3)}
HOMOSEX2ID = {c:i for i,c in enumerate(HOMOSEX4)}
K = len(HOMOSEX4)

YEARS_CS = list(range(2006, 2025, 2))  # 2006..2024 every 2 years

In [23]:
# EXPERIMENT SETUP
EXPERIMENTS = [
    {"name": "exp_A_train_le_2022", "TRAIN_MAX_YEAR": 2022, "FORECAST_YEARS": [2024]},
    {"name": "exp_B_train_le_2018", "TRAIN_MAX_YEAR": 2018, "FORECAST_YEARS": [2020, 2022, 2024]},
    {"name": "exp_C_train_le_2010", "TRAIN_MAX_YEAR": 2010, "FORECAST_YEARS": [2012, 2014, 2016, 2018, 2020, 2022, 2024]},
]

In [24]:
# Load and harmonize the data
# --- Cross-sectional ---
cs = pd.read_csv(CS_CSV)

# Convert year 2021 to 2020
cs["year"] = cs["year"].apply(lambda x: 2020 if x==2021 else x)

# Expect: yearid, year, abortion_att4, generation, race, gender, edu_level, wtssps
# Map the attitude to canonical
# cs["att"] = cs["abortion_att4"].astype(str).str.strip()
# keep only canon categories, drop NAs
# cs_abortion = cs[cs["att"].isin(ABORT4)].copy()
# cs_abortion["wt"] = cs_abortion.get("wtssps", pd.Series([1.0]*len(cs_abortion)))  # default 1.0 if missing
# cs_trust = cs[cs["trust"].astype(str).str.strip().isin(TRUST3)].copy()
# cs_trust['wt'] = cs_trust.get("wtssps", pd.Series([1.0]*len(cs_trust)))  # default 1.0 if missing

# cs_envir = cs[cs["natenvir"].astype(str).str.strip().isin(ENVIR3)].copy()
# cs_envir['wt'] = cs_envir.get("wtssps", pd.Series([1.0]*len(cs_envir)))  # default 1.0 if missing

cs_homosex = cs[cs["homosex"].astype(str).str.strip().isin(HOMOSEX4)].copy()
cs_homosex['wt'] = cs_homosex.get("wtssps", pd.Series([1.0]*len(cs_homosex)))  # default 1.0 if missing

# --- Panel ---
pl = pd.read_csv(PANEL_CSV)
# Convert year 2021 to 2020
pl["year"] = pl["year"].apply(lambda x: 2020 if x==2021 else x)
# pl["att"] = pl["abortion_att4"].astype(str).str.strip()
# pl_abortion = pl[pl["att"].isin(ABORT4)].copy()
# pl_trust = pl[pl["trust"].astype(str).str.strip().isin(TRUST3)].copy()
# pl_envir = pl[pl["natenvir"].astype(str).str.strip().isin(ENVIR3)].copy()
pl_homosex = pl[pl["homosex"].astype(str).str.strip().isin(HOMOSEX4)].copy()

# Define grouping keys
GROUP_COLS_4 = ["generation","gender","race","edu_level"]
GROUP_COLS_3 = ["generation","gender","race"]
GROUP_COLS_2 = ["generation","gender"]
GROUP_COLS_1 = ["generation"]

for df in (cs, pl):
    for c in GROUP_COLS_4:
        df[c] = df[c].astype(str).str.strip()

# Keep original cs_homosex and pl_homosex as FULL
cs_full = cs_homosex.copy()
pl_full = pl_homosex.copy()

def make_filtered_views(TRAIN_MAX_YEAR):
    cs_train = cs_full[cs_full["year"] <= TRAIN_MAX_YEAR].copy()
    pl_train = pl_full[pl_full["year"] <= TRAIN_MAX_YEAR].copy()
    return cs_train, pl_train

In [12]:
# Build cross-section margins p_cs[g,y] (weighted)
def group_key(row):
    return (row["generation"], row["gender"], row["race"], row['edu_level'])

def weighted_probs(vals, wts, cats=HOMOSEX4):
    # vals: list of category strings; wts: weights
    counts = {c:0.0 for c in cats}
    for v, w in zip(vals, wts):
        counts[v] += float(w)
    vec = np.array([counts[c] for c in cats], dtype=float)
    s = vec.sum()
    if s <= 0: return None
    return vec / s

In [13]:
YEARS_CS

[2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024]

In [14]:
def build_p_cs_from(df_cs, group_cols=GROUP_COLS_4, cats=HOMOSEX4):
    p_cs, effN_cs = {}, {}
    for y, sub in df_cs.groupby("year"):
        for g_vals, df_g in sub.groupby(group_cols):
            p = weighted_probs(df_g["homosex"].tolist(), df_g["wt"].tolist(), cats)
            if p is None:
                continue
            p_cs[(g_vals, int(y))] = p
            effN_cs[(g_vals, int(y))] = float(df_g["wt"].sum())
    return p_cs, effN_cs

# Also keep the FULL p_cs for later evaluation (observed margins)
p_cs_full, effN_cs_full = build_p_cs_from(cs_full, GROUP_COLS_4, HOMOSEX4)

In [15]:
# JSONL builders: Task A (panel rows) and Task B (margins)
def smooth_row(row_counts, alpha):
    rc = np.array(row_counts, dtype=float) + alpha
    s = rc.sum()
    if s <= 0:
        return np.ones_like(rc)/len(rc)
    return rc / s

def build_taskA_rows_timegated(C, Nfrom, p_cs, out_jsonl, TRAIN_MAX_YEAR,
                               alpha_small=0.05, alpha_big=0.25, n_thresh=20, cap_weight=100.0):
    out = open(out_jsonl, "w", encoding="utf-8")
    n_rows = 0
    for (g,t,Δ), mat in C.items():
        t1 = t + Δ
        if t1 > TRAIN_MAX_YEAR:             # ---- time gate ----
            continue
        nrow = Nfrom[(g,t,Δ)]
        for i in range(K):
            alpha = alpha_small if nrow[i] >= n_thresh else alpha_big
            tgt = smooth_row(mat[i,:], alpha=alpha).tolist()
            prompt = (
                "[Task: Predict transition row]\n"
                f"From: <Y{t}> → To: <Y{t1}> <DT{Δ}>\n"
                f"Group: generation={g[0]}; gender={g[1]}; race={g[2]}; edu_level={g[3]}\n"
                f"From option: {HOMOSEX4[i]}\n"
                "Answer:\n"
            )
            rec = {
                "task": "row",
                "group": {"generation": g[0], "gender": g[1], "race": g[2], "edu_level": g[3]},
                "year_t": t, "year_t1": t1, "dt": Δ,
                "from_bin": HOMOSEX4[i],
                "prompt_text": prompt,
                "to_dist": tgt,
                "weight": float(min(nrow[i], cap_weight))
            }
            # attach margins if available
            if ((g,t) in p_cs) and ((g,t1) in p_cs):
                rec["p_curr"] = p_cs[(g,t)].tolist()
                rec["p_next"] = p_cs[(g,t1)].tolist()
            out.write(json.dumps(rec) + "\n")
            n_rows += 1
    out.close()
    return n_rows

def build_taskB_rows_timegated(p_cs, effN_cs, out_jsonl, TRAIN_MAX_YEAR, lags=(2,4), cap_weight=500.0):
    out = open(out_jsonl, "w", encoding="utf-8")
    n_rows = 0
    by_g = {}
    for (g,y) in p_cs.keys():
        by_g.setdefault(g, []).append(y)
    for g, years in by_g.items():
        ys = sorted(years)
        for y in ys:
            if y > TRAIN_MAX_YEAR:          # ---- time gate ----
                continue
            ctx = []
            for L in lags:
                yprev = y - L
                if (g, yprev) in p_cs:
                    ctx.append((yprev, p_cs[(g, yprev)]))
            if len(ctx) == 0:
                continue
            ctx_parts = " ".join([f"<Y{yy}>[{','.join(f'{x:.4f}' for x in p)}]" for (yy,p) in ctx])
            prompt = (
                "[Task: Forecast next-wave margin]\n"
                f"Group: generation={g[0]}; gender={g[1]}; race={g[2]}; edu_level={g[3]}\n"
                f"Context: {ctx_parts}\n"
                f"Predict: <Y{y}>\n"
                "Answer:\n"
            )
            w = float(min(effN_cs.get((g,y), 1.0), cap_weight))
            rec = {
                "task": "margin",
                "group": {"generation": g[0], "gender": g[1], "race": g[2], "edu_level": g[3]},
                "year": y,
                "prompt_text": prompt,
                "to_dist": p_cs[(g,y)].tolist(),
                "weight": w
            }
            out.write(json.dumps(rec) + "\n")
            n_rows += 1
    out.close()
    return n_rows


In [16]:
def build_panel_C_from(pl_df, group_cols=GROUP_COLS_4):
    from collections import defaultdict
    C = defaultdict(lambda: np.zeros((K,K), dtype=float))
    Nfrom = defaultdict(lambda: np.zeros((K,), dtype=float))

    def canon_index(cat): return HOMOSEX2ID.get(cat, None)

    tmp = pl_df.copy()
    tmp["w"] = 1.0
    for pid, df_id in tmp.groupby("yearid"):
        df_id = df_id.sort_values("year").drop_duplicates(subset=["year"], keep="last")
        years = df_id["year"].values.tolist()
        vals  = df_id["homosex"].values.tolist()
        wgts  = df_id["w"].values.astype(float).tolist()
        gens  = df_id["generation"].values.tolist()
        gend  = df_id["gender"].values.tolist()
        race  = df_id["race"].values.tolist()
        edu   = df_id["edu_level"].values.tolist()
        # require constant group labels
        if not (len(set(gens))==1 and len(set(gend))==1 and len(set(race))==1 and len(set(edu))==1):
            continue
        g = (gens[0], gend[0], race[0], edu[0])
        for i in range(len(years)-1):
            t, t1 = int(years[i]), int(years[i+1])
            Δ = t1 - t
            if Δ <= 0:
                continue
            ai, aj = canon_index(vals[i]), canon_index(vals[i+1])
            if ai is None or aj is None:
                continue
            w = float(wgts[i])
            C[(g,t,Δ)][ai, aj] += w
            Nfrom[(g,t,Δ)][ai] += w
    return C, Nfrom

In [17]:
# Datasets and Collator (multitask)
class MTJsonlDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_paths: List[str], tokenizer, max_len=768):
        self.rows = []
        for p in jsonl_paths:
            with open(p, "r", encoding="utf-8") as f:
                self.rows.extend([json.loads(x) for x in f])
        random.shuffle(self.rows)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        enc = self.tok(r["prompt_text"], return_tensors="pt", truncation=True, max_length=self.max_len)
        out = {
            "task_type": r["task"],
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "to_dist": torch.tensor(r["to_dist"], dtype=torch.float),
            "weight": torch.tensor(float(r.get("weight", 1.0)), dtype=torch.float),
        }
        # add optional consistency fields
        if r["task"]=="row" and ("p_curr" in r) and ("p_next" in r):
            out["p_curr"] = torch.tensor(r["p_curr"], dtype=torch.float)
            out["p_next"] = torch.tensor(r["p_next"], dtype=torch.float)
            out["has_consistency"] = torch.tensor(1, dtype=torch.long)
        else:
            out["has_consistency"] = torch.tensor(0, dtype=torch.long)
        return out

# Model: LLM backbone + two small heads + last-K pooling
# Tokenizer & backbone
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def mt_collate(batch):
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    maxlen = max(x["input_ids"].shape[0] for x in batch)

    def pad(seq, pad_val, target_len):
        pad_n = target_len - seq.shape[0]
        if pad_n <= 0: return seq
        return torch.cat([seq, torch.full((pad_n,), pad_val, dtype=seq.dtype)])

    input_ids      = torch.stack([pad(x["input_ids"], pad_id, maxlen) for x in batch])
    attention_mask = torch.stack([pad(x["attention_mask"], 0, maxlen) for x in batch])
    to_dist        = torch.stack([x["to_dist"] for x in batch])
    weight         = torch.stack([x["weight"] for x in batch])
    has_cons       = torch.stack([x["has_consistency"] for x in batch])
    # p_curr/p_next if present; else zeros
    p_curr = torch.zeros(len(batch), K, dtype=torch.float)
    p_next = torch.zeros(len(batch), K, dtype=torch.float)
    for i, x in enumerate(batch):
        if x["has_consistency"]==1:
            p_curr[i] = x["p_curr"]
            p_next[i] = x["p_next"]

    task_types = [x["task_type"] for x in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "to_dist": to_dist,
        "weight": weight,
        "has_consistency": has_cons,
        "p_curr": p_curr,
        "p_next": p_next,
        "task_types": task_types,
    }

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    output_hidden_states=True
)

# Add LoRA to attention/MLP projections
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj"]
)
model = get_peft_model(base, lora_cfg)

# Two task heads
class TwoHead(nn.Module):
    def __init__(self, hidden_size, K):
        super().__init__()
        self.head_row    = nn.Linear(hidden_size, K)   # Task A
        self.head_margin = nn.Linear(hidden_size, K)   # Task B

    def forward(self, feats):
        return self.head_row(feats), self.head_margin(feats)

hidden_size = base.config.hidden_size
two_head = TwoHead(hidden_size, K).to(model.device)

# Simple pooled features: mean of last K tokens (tail window)
def pooled_features(outputs, attention_mask, tail=96):
    hs = outputs.hidden_states[-1]     # [B,T,H]
    B, T, H = hs.shape
    valid_lens = attention_mask.sum(dim=1)  # [B]
    feats = []
    for b in range(B):
        L = int(valid_lens[b].item())
        s = max(0, L - tail); e = L
        if e <= s: s, e = max(0, L-32), L
        feats.append(hs[b, s:e, :].mean(dim=0))
    feats = torch.stack(feats, dim=0)
    return feats

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [18]:
# Trainer with multitask losses (KL + optional consistency)
class MTTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        self.two_head = kwargs.pop("two_head")
        self.lambda_A = kwargs.pop("lambda_A", 1.0)
        self.lambda_B = kwargs.pop("lambda_B", 1.0)
        self.lambda_C = kwargs.pop("lambda_C", 0.5)
        super().__init__(*args, **kwargs)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs: bool = False,
        num_items_in_batch: Optional[int] = None,  # <-- accept the kwarg
    ):
        input_ids      = inputs["input_ids"].to(model.device)
        attention_mask = inputs["attention_mask"].to(model.device)
        to_dist        = inputs["to_dist"].to(model.device)     # [B,K]
        weight         = inputs["weight"].to(model.device)      # [B]
        has_cons       = inputs["has_consistency"].to(model.device)  # [B]
        p_curr         = inputs["p_curr"].to(model.device)      # [B,K]
        p_next         = inputs["p_next"].to(model.device)      # [B,K]
        task_types     = inputs["task_types"]                   # list[str]

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            out = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        feats = pooled_features(out, attention_mask, tail=96).to(model.device)  # [B,H]

        logits_row, logits_margin = self.two_head(feats)   # [B,K], [B,K]
        p_row_hat    = F.softmax(logits_row, dim=1)
        p_margin_hat = F.softmax(logits_margin, dim=1)

        # Build masks by task
        is_row    = torch.tensor([t == "row"    for t in task_types], device=model.device, dtype=torch.bool)
        is_margin = torch.tensor([t == "margin" for t in task_types], device=model.device, dtype=torch.bool)

        eps = 1e-8
        def fwd_kl(p, q):
            p = p.clamp_min(eps); q = q.clamp_min(eps)
            return (p * (p.log() - q.log())).sum(dim=1)

        loss = torch.tensor(0.0, device=model.device)

        # Task A: panel rows
        if is_row.any():
            L_A = fwd_kl(to_dist[is_row], p_row_hat[is_row])
            loss = loss + self.lambda_A * (weight[is_row] * L_A).mean()

        # Task B: margins
        if is_margin.any():
            L_B = fwd_kl(to_dist[is_margin], p_margin_hat[is_margin])
            loss = loss + self.lambda_B * (weight[is_margin] * L_B).mean()

        # Consistency (disabled unless you wire full-row assembly):
        # cons_mask = is_row & (has_cons==1)
        # if cons_mask.any() and self.lambda_C > 0:
        #     ...

        if return_outputs:
            return loss, {"logits_row": logits_row, "logits_margin": logits_margin, "labels": to_dist}
        else:
            return loss

In [19]:
class SubsetDS(torch.utils.data.Dataset):
    def __init__(self, base, keep_idx):
        self.base = base
        self.keep = sorted(list(keep_idx))
    def __len__(self): return len(self.keep)
    def __getitem__(self, i): return self.base[self.keep[i]]

In [20]:
for exp in EXPERIMENTS:
    name = exp["name"]
    TRAIN_MAX_YEAR = exp["TRAIN_MAX_YEAR"]
    FORECAST_YEARS = exp["FORECAST_YEARS"]

    print(f"\n=== {name}: train ≤ {TRAIN_MAX_YEAR} ===")
    out_dir_exp = os.path.join(OUT_DIR, name)
    os.makedirs(out_dir_exp, exist_ok=True)

    # 1) filtered views
    cs_train, pl_train = make_filtered_views(TRAIN_MAX_YEAR)

    # 2) training p_cs from filtered CS
    p_cs_train, effN_cs_train = build_p_cs_from(cs_train, GROUP_COLS_4, HOMOSEX4)

    # 3) panel transitions from filtered panel
    C_train, Nfrom_train = build_panel_C_from(pl_train, GROUP_COLS_4)

    # 4) write JSONLs (time-gated inside)
    TASKA_JSONL = os.path.join(out_dir_exp, f"taskA_rows_le_{TRAIN_MAX_YEAR}.jsonl")
    TASKB_JSONL = os.path.join(out_dir_exp, f"taskB_margins_le_{TRAIN_MAX_YEAR}.jsonl")
    na = build_taskA_rows_timegated(C_train, Nfrom_train, p_cs_train, TASKA_JSONL, TRAIN_MAX_YEAR)
    nb = build_taskB_rows_timegated(p_cs_train, effN_cs_train, TASKB_JSONL, TRAIN_MAX_YEAR)
    print(f"Wrote Task A rows: {na}  |  Task B rows: {nb}")

    # 5) make datasets / splits
    train_ds = MTJsonlDataset([TASKA_JSONL, TASKB_JSONL], tokenizer, max_len=768)
    # small time-based split (optional: use the last 10% by prompt year as val)
    val_frac = 0.1
    n_val = max(1, int(len(train_ds) * val_frac))
    idx = list(range(len(train_ds)))
    random.seed(0); random.shuffle(idx)
    val_idx  = set(idx[:n_val])
    train_idx= set(idx[n_val:])
    ds_train = SubsetDS(train_ds, train_idx)
    ds_val   = SubsetDS(train_ds, val_idx)

    # 6) train
    args = TrainingArguments(
        output_dir=os.path.join(out_dir_exp, "ckpt"),
        learning_rate=1e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        num_train_epochs=20,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_steps=200,
        save_total_limit=2,
        bf16=True,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        warmup_steps=300,
        weight_decay=0.0,
        report_to="none",
        remove_unused_columns=False,
    )
    # fresh heads each experiment
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        output_hidden_states=True
    )
    model = get_peft_model(base, lora_cfg)
    two_head = TwoHead(hidden_size=base.config.hidden_size, K=K).to(model.device)

    trainer = MTTrainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        data_collator=mt_collate,
        two_head=two_head,
        lambda_A=1.0, lambda_B=1.0, lambda_C=0.0,
    )
    trainer.train()

    # 7) save per-experiment
    save_dir = os.path.join(out_dir_exp, "final_ckpt")
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    torch.save(two_head.state_dict(), os.path.join(save_dir, f"two_head_{name}.pt"))
    print("Saved:", save_dir)



=== exp_A_train_le_2022: train ≤ 2022 ===
Wrote Task A rows: 1476  |  Task B rows: 548


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


Step,Training Loss,Validation Loss
200,3.794300,No log
400,3.400900,No log
600,2.489200,No log
800,2.419000,No log
1000,2.906200,No log
1200,1.770000,No log
1400,1.993000,No log
1600,1.737600,No log
1800,1.410400,No log
2000,1.424300,No log


/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', arg

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


Step,Training Loss,Validation Loss
200,3.812600,No log
400,2.700900,No log
600,2.349500,No log
800,1.487900,No log
1000,0.999900,No log
1200,0.578800,No log
1400,0.438700,No log


/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', arg

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_p_trial/exp_C_train_le_2010/final_ckpt


### HOMOSEX, Group 1

In [25]:
# Set up paths
OUT_DIR   = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_r_trial_homosex_g1"          # where I write JSONLs & checkpoints
os.makedirs(OUT_DIR, exist_ok=True)

In [26]:
# Build cross-section margins p_cs[g,y] (weighted)
def group_key(row):
    return (row["generation"])

def weighted_probs(vals, wts, cats=HOMOSEX4):
    # vals: list of category strings; wts: weights
    counts = {c:0.0 for c in cats}
    for v, w in zip(vals, wts):
        counts[v] += float(w)
    vec = np.array([counts[c] for c in cats], dtype=float)
    s = vec.sum()
    if s <= 0: return None
    return vec / s

In [27]:
def build_p_cs_from(df_cs, group_cols=GROUP_COLS_1, cats=HOMOSEX4):
    p_cs, effN_cs = {}, {}
    for y, sub in df_cs.groupby("year"):
        for g_vals, df_g in sub.groupby(group_cols):
            p = weighted_probs(df_g["homosex"].tolist(), df_g["wt"].tolist(), cats)
            if p is None:
                continue
            p_cs[(g_vals, int(y))] = p
            effN_cs[(g_vals, int(y))] = float(df_g["wt"].sum())
    return p_cs, effN_cs

# Also keep the FULL p_cs for later evaluation (observed margins)
p_cs_full, effN_cs_full = build_p_cs_from(cs_full, GROUP_COLS_1, HOMOSEX4)

In [34]:
# JSONL builders: Task A (panel rows) and Task B (margins)
def smooth_row(row_counts, alpha):
    rc = np.array(row_counts, dtype=float) + alpha
    s = rc.sum()
    if s <= 0:
        return np.ones_like(rc)/len(rc)
    return rc / s

def build_taskA_rows_timegated(C, Nfrom, p_cs, out_jsonl, TRAIN_MAX_YEAR,
                               alpha_small=0.05, alpha_big=0.25, n_thresh=20, cap_weight=100.0):
    out = open(out_jsonl, "w", encoding="utf-8")
    n_rows = 0
    for (g_vals,t,Δ), mat in C.items():
        t1 = t + Δ
        if t1 > TRAIN_MAX_YEAR:             # ---- time gate ----
            continue
        nrow = Nfrom[(g_vals,t,Δ)]
        # Ensure g is a tuple for consistent access
        if isinstance(g_vals, (str, int, float)):
            g = (g_vals,)
        else:
            g = g_vals

        for i in range(K):
            alpha = alpha_small if nrow[i] >= n_thresh else alpha_big
            tgt = smooth_row(mat[i,:], alpha=alpha).tolist()
            prompt = (
                "[Task: Predict transition row]\n"
                f"From: <Y{t}> → To: <Y{t1}> <DT{Δ}>\n"
                f"Group: generation={g[0]}\n"
                f"From option: {HOMOSEX4[i]}\n"
                "Answer:\n"
            )
            rec = {
                "task": "row",
                "group": {"generation": g[0]},
                "year_t": t, "year_t1": t1, "dt": Δ,
                "from_bin": HOMOSEX4[i],
                "prompt_text": prompt,
                "to_dist": tgt,
                "weight": float(min(nrow[i], cap_weight))
            }
            # attach margins if available
            if ((g_vals,t) in p_cs) and ((g_vals,t1) in p_cs):
                rec["p_curr"] = p_cs[(g_vals,t)].tolist()
                rec["p_next"] = p_cs[(g_vals,t1)].tolist()
            out.write(json.dumps(rec) + "\n")
            n_rows += 1
    out.close()
    return n_rows

def build_taskB_rows_timegated(p_cs, effN_cs, out_jsonl, TRAIN_MAX_YEAR, lags=(2,4), cap_weight=500.0):
    out = open(out_jsonl, "w", encoding="utf-8")
    n_rows = 0
    by_g = {}
    for (g_vals,y) in p_cs.keys():
        by_g.setdefault(g_vals, []).append(y)
    for g_vals, years in by_g.items():
        # Ensure g is a tuple for consistent access
        if isinstance(g_vals, (str, int, float)):
            g = (g_vals,)
        else:
            g = g_vals

        ys = sorted(years)
        for y in ys:
            if y > TRAIN_MAX_YEAR:          # ---- time gate ----
                continue
            ctx = []
            for L in lags:
                yprev = y - L
                if (g_vals, yprev) in p_cs:
                    ctx.append((yprev, p_cs[(g_vals, yprev)]))
            if len(ctx) == 0:
                continue
            ctx_parts = " ".join([f"<Y{yy}>[{','.join(f'{x:.4f}' for x in p)}]" for (yy,p) in ctx])
            prompt = (
                "[Task: Forecast next-wave margin]\n"
                f"Group: generation={g[0]}\n"
                f"Context: {ctx_parts}\n"
                f"Predict: <Y{y}>\n"
                "Answer:\n"
            )
            w = float(min(effN_cs.get((g_vals,y), 1.0), cap_weight))
            rec = {
                "task": "margin",
                "group": {"generation": g[0]},
                "year": y,
                "prompt_text": prompt,
                "to_dist": p_cs[(g_vals,y)].tolist(),
                "weight": w
            }
            out.write(json.dumps(rec) + "\n")
            n_rows += 1
    out.close()
    return n_rows

In [35]:
def build_panel_C_from(pl_df, group_cols=GROUP_COLS_1):
    from collections import defaultdict
    C = defaultdict(lambda: np.zeros((K,K), dtype=float))
    Nfrom = defaultdict(lambda: np.zeros((K,), dtype=float))

    def canon_index(cat): return HOMOSEX2ID.get(cat, None)

    tmp = pl_df.copy()
    tmp["w"] = 1.0
    for pid, df_id in tmp.groupby("yearid"):
        df_id = df_id.sort_values("year").drop_duplicates(subset=["year"], keep="last")
        years = df_id["year"].values.tolist()
        vals  = df_id["homosex"].values.tolist()
        wgts  = df_id["w"].values.astype(float).tolist()
        gens  = df_id["generation"].values.tolist()
        # gend  = df_id["gender"].values.tolist()
        # race  = df_id["race"].values.tolist()
        # edu   = df_id["edu_level"].values.tolist()
        # require constant group labels
        if not (len(set(gens))==1):
            continue
        g = (gens[0])
        for i in range(len(years)-1):
            t, t1 = int(years[i]), int(years[i+1])
            Δ = t1 - t
            if Δ <= 0:
                continue
            ai, aj = canon_index(vals[i]), canon_index(vals[i+1])
            if ai is None or aj is None:
                continue
            w = float(wgts[i])
            C[(g,t,Δ)][ai, aj] += w
            Nfrom[(g,t,Δ)][ai] += w
    return C, Nfrom

In [36]:
# Datasets and Collator (multitask)
class MTJsonlDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_paths: List[str], tokenizer, max_len=768):
        self.rows = []
        for p in jsonl_paths:
            with open(p, "r", encoding="utf-8") as f:
                self.rows.extend([json.loads(x) for x in f])
        random.shuffle(self.rows)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        enc = self.tok(r["prompt_text"], return_tensors="pt", truncation=True, max_length=self.max_len)
        out = {
            "task_type": r["task"],
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "to_dist": torch.tensor(r["to_dist"], dtype=torch.float),
            "weight": torch.tensor(float(r.get("weight", 1.0)), dtype=torch.float),
        }
        # add optional consistency fields
        if r["task"]=="row" and ("p_curr" in r) and ("p_next" in r):
            out["p_curr"] = torch.tensor(r["p_curr"], dtype=torch.float)
            out["p_next"] = torch.tensor(r["p_next"], dtype=torch.float)
            out["has_consistency"] = torch.tensor(1, dtype=torch.long)
        else:
            out["has_consistency"] = torch.tensor(0, dtype=torch.long)
        return out

# Model: LLM backbone + two small heads + last-K pooling
# Tokenizer & backbone
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def mt_collate(batch):
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    maxlen = max(x["input_ids"].shape[0] for x in batch)

    def pad(seq, pad_val, target_len):
        pad_n = target_len - seq.shape[0]
        if pad_n <= 0: return seq
        return torch.cat([seq, torch.full((pad_n,), pad_val, dtype=seq.dtype)])

    input_ids      = torch.stack([pad(x["input_ids"], pad_id, maxlen) for x in batch])
    attention_mask = torch.stack([pad(x["attention_mask"], 0, maxlen) for x in batch])
    to_dist        = torch.stack([x["to_dist"] for x in batch])
    weight         = torch.stack([x["weight"] for x in batch])
    has_cons       = torch.stack([x["has_consistency"] for x in batch])
    # p_curr/p_next if present; else zeros
    p_curr = torch.zeros(len(batch), K, dtype=torch.float)
    p_next = torch.zeros(len(batch), K, dtype=torch.float)
    for i, x in enumerate(batch):
        if x["has_consistency"]==1:
            p_curr[i] = x["p_curr"]
            p_next[i] = x["p_next"]

    task_types = [x["task_type"] for x in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "to_dist": to_dist,
        "weight": weight,
        "has_consistency": has_cons,
        "p_curr": p_curr,
        "p_next": p_next,
        "task_types": task_types,
    }

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    output_hidden_states=True
)

# Add LoRA to attention/MLP projections
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj"]
)
model = get_peft_model(base, lora_cfg)

# Two task heads
class TwoHead(nn.Module):
    def __init__(self, hidden_size, K):
        super().__init__()
        self.head_row    = nn.Linear(hidden_size, K)   # Task A
        self.head_margin = nn.Linear(hidden_size, K)   # Task B

    def forward(self, feats):
        return self.head_row(feats), self.head_margin(feats)

hidden_size = base.config.hidden_size
two_head = TwoHead(hidden_size, K).to(model.device)

# Simple pooled features: mean of last K tokens (tail window)
def pooled_features(outputs, attention_mask, tail=96):
    hs = outputs.hidden_states[-1]     # [B,T,H]
    B, T, H = hs.shape
    valid_lens = attention_mask.sum(dim=1)  # [B]
    feats = []
    for b in range(B):
        L = int(valid_lens[b].item())
        s = max(0, L - tail); e = L
        if e <= s: s, e = max(0, L-32), L
        feats.append(hs[b, s:e, :].mean(dim=0))
    feats = torch.stack(feats, dim=0)
    return feats

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [37]:
# Trainer with multitask losses (KL + optional consistency)
class MTTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        self.two_head = kwargs.pop("two_head")
        self.lambda_A = kwargs.pop("lambda_A", 1.0)
        self.lambda_B = kwargs.pop("lambda_B", 1.0)
        self.lambda_C = kwargs.pop("lambda_C", 0.5)
        super().__init__(*args, **kwargs)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs: bool = False,
        num_items_in_batch: Optional[int] = None,  # <-- accept the kwarg
    ):
        input_ids      = inputs["input_ids"].to(model.device)
        attention_mask = inputs["attention_mask"].to(model.device)
        to_dist        = inputs["to_dist"].to(model.device)     # [B,K]
        weight         = inputs["weight"].to(model.device)      # [B]
        has_cons       = inputs["has_consistency"].to(model.device)  # [B]
        p_curr         = inputs["p_curr"].to(model.device)      # [B,K]
        p_next         = inputs["p_next"].to(model.device)      # [B,K]
        task_types     = inputs["task_types"]                   # list[str]

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            out = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        feats = pooled_features(out, attention_mask, tail=96).to(model.device)  # [B,H]

        logits_row, logits_margin = self.two_head(feats)   # [B,K], [B,K]
        p_row_hat    = F.softmax(logits_row, dim=1)
        p_margin_hat = F.softmax(logits_margin, dim=1)

        # Build masks by task
        is_row    = torch.tensor([t == "row"    for t in task_types], device=model.device, dtype=torch.bool)
        is_margin = torch.tensor([t == "margin" for t in task_types], device=model.device, dtype=torch.bool)

        eps = 1e-8
        def fwd_kl(p, q):
            p = p.clamp_min(eps); q = q.clamp_min(eps)
            return (p * (p.log() - q.log())).sum(dim=1)

        loss = torch.tensor(0.0, device=model.device)

        # Task A: panel rows
        if is_row.any():
            L_A = fwd_kl(to_dist[is_row], p_row_hat[is_row])
            loss = loss + self.lambda_A * (weight[is_row] * L_A).mean()

        # Task B: margins
        if is_margin.any():
            L_B = fwd_kl(to_dist[is_margin], p_margin_hat[is_margin])
            loss = loss + self.lambda_B * (weight[is_margin] * L_B).mean()

        # Consistency (disabled unless you wire full-row assembly):
        # cons_mask = is_row & (has_cons==1)
        # if cons_mask.any() and self.lambda_C > 0:
        #     ...

        if return_outputs:
            return loss, {"logits_row": logits_row, "logits_margin": logits_margin, "labels": to_dist}
        else:
            return loss

In [38]:
class SubsetDS(torch.utils.data.Dataset):
    def __init__(self, base, keep_idx):
        self.base = base
        self.keep = sorted(list(keep_idx))
    def __len__(self): return len(self.keep)
    def __getitem__(self, i): return self.base[self.keep[i]]

In [39]:
for exp in EXPERIMENTS:
    name = exp["name"]
    TRAIN_MAX_YEAR = exp["TRAIN_MAX_YEAR"]
    FORECAST_YEARS = exp["FORECAST_YEARS"]

    print(f"\n=== {name}: train ≤ {TRAIN_MAX_YEAR} ===")
    out_dir_exp = os.path.join(OUT_DIR, name)
    os.makedirs(out_dir_exp, exist_ok=True)

    # 1) filtered views
    cs_train, pl_train = make_filtered_views(TRAIN_MAX_YEAR)

    # 2) training p_cs from filtered CS
    p_cs_train, effN_cs_train = build_p_cs_from(cs_train, GROUP_COLS_1, HOMOSEX4)

    # 3) panel transitions from filtered panel
    C_train, Nfrom_train = build_panel_C_from(pl_train, GROUP_COLS_1)

    # 4) write JSONLs (time-gated inside)
    TASKA_JSONL = os.path.join(out_dir_exp, f"taskA_rows_le_{TRAIN_MAX_YEAR}.jsonl")
    TASKB_JSONL = os.path.join(out_dir_exp, f"taskB_margins_le_{TRAIN_MAX_YEAR}.jsonl")
    na = build_taskA_rows_timegated(C_train, Nfrom_train, p_cs_train, TASKA_JSONL, TRAIN_MAX_YEAR)
    nb = build_taskB_rows_timegated(p_cs_train, effN_cs_train, TASKB_JSONL, TRAIN_MAX_YEAR)
    print(f"Wrote Task A rows: {na}  |  Task B rows: {nb}")

    # 5) make datasets / splits
    train_ds = MTJsonlDataset([TASKA_JSONL, TASKB_JSONL], tokenizer, max_len=768)
    # small time-based split (optional: use the last 10% by prompt year as val)
    val_frac = 0.1
    n_val = max(1, int(len(train_ds) * val_frac))
    idx = list(range(len(train_ds)))
    random.seed(0); random.shuffle(idx)
    val_idx  = set(idx[:n_val])
    train_idx= set(idx[n_val:])
    ds_train = SubsetDS(train_ds, train_idx)
    ds_val   = SubsetDS(train_ds, val_idx)

    # 6) train
    args = TrainingArguments(
        output_dir=os.path.join(out_dir_exp, "ckpt"),
        learning_rate=1e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        num_train_epochs=20,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_steps=200,
        save_total_limit=2,
        bf16=True,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        warmup_steps=300,
        weight_decay=0.0,
        report_to="none",
        remove_unused_columns=False,
    )
    # fresh heads each experiment
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        output_hidden_states=True
    )
    model = get_peft_model(base, lora_cfg)
    two_head = TwoHead(hidden_size=base.config.hidden_size, K=K).to(model.device)

    trainer = MTTrainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        data_collator=mt_collate,
        two_head=two_head,
        lambda_A=1.0, lambda_B=1.0, lambda_C=0.0,
    )
    trainer.train()

    # 7) save per-experiment
    save_dir = os.path.join(out_dir_exp, "final_ckpt")
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    torch.save(two_head.state_dict(), os.path.join(save_dir, f"two_head_{name}.pt"))
    print("Saved:", save_dir)


=== exp_A_train_le_2022: train ≤ 2022 ===
Wrote Task A rows: 156  |  Task B rows: 35


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


Step,Training Loss,Validation Loss
200,10.481300,No log
400,4.118500,No log


/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


...

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_r_trial_homosex_g1/exp_B_train_le_2018/final_ckpt

=== exp_C_train_le_2010: train ≤ 2010 ===
Wrote Task A rows: 56  |  Task B rows: 8


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
/tmp/ipython-input-4226314306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


Step,Training Loss,Validation Loss


Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_r_trial_homosex_g1/exp_C_train_le_2010/final_ckpt
